In [1]:
!pip install bert_score

import os
import re
import gc
import string
import pandas as pd
import torch
from PIL import Image
from tqdm.notebook import tqdm
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from transformers import AutoProcessor, Blip2ForConditionalGeneration
from nltk.corpus import stopwords

# ─── NLTK Downloads ───────────────────────────────────────────────────
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

# ─── Global Stopword Set ──────────────────────────────────────────────
STOPWORDS = set(stopwords.words('english'))

# ─── Load BLIP-2 Model ────────────────────────────────────────────────
def load_model():
    processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16
    ).to("cuda")
    return processor, model

# ─── Text Cleaning ────────────────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().replace('_', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

# ─── Answer Extraction ────────────────────────────────────────────────
def extract_answer(text, question):
    if not text:
        return ""
    if isinstance(text, list):
        text = text[0]
    if "Answer:" in text:
        answer = text.split("Answer:")[1].strip()
    else:
        question_prefix = f"Question: {question}"
        answer = text.replace(question_prefix, "").strip() if question_prefix in text else text.strip()

    if question.lower().startswith(("is ", "are ", "does ", "can ", "was ", "will ", "should ")):
        if answer.lower().startswith("yes"):
            return "yes"
        elif answer.lower().startswith("no"):
            return "no"
    return answer

# ─── Inference Loop ───────────────────────────────────────────────────
def process_dataset(df, base_dir, processor, model):
    predictions = []
    raw_predictions = []
    cleaned_predictions = []
    cleaned_ground_truths = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            image = Image.open(os.path.join(base_dir, row['path'])).convert('RGB')
            question = row['generated_question']
            ground_truth = row['generated_answer']

            prompt = f"Question: {question} Answer:"
            inputs = processor(image, text=prompt, return_tensors="pt").to("cuda", torch.float16)

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=10)

            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
            raw_answer = extract_answer(generated_text, question)
            cleaned_answer = clean_text(raw_answer)
            cleaned_gt = clean_text(ground_truth)

            predictions.append(raw_answer)
            raw_predictions.append(generated_text)
            cleaned_predictions.append(cleaned_answer)
            cleaned_ground_truths.append(cleaned_gt)

            if idx % 50 == 0:
                print(f"\n[{idx}] Q: {question}")
                print(f"Pred: {cleaned_answer} | GT: {cleaned_gt}")

        except Exception as e:
            print(f"Error @ {row['path']}: {e}")
            predictions.append("error")
            raw_predictions.append("error")
            cleaned_predictions.append("error")
            cleaned_ground_truths.append("error")

    return predictions, raw_predictions, cleaned_predictions, cleaned_ground_truths

# ─── Partial Match: pred in truth or truth in pred ────────────────────
def is_partial_match(pred, truth):
    return pred in truth or truth in pred

# ─── Metric Evaluation ────────────────────────────────────────────────
def evaluate_metrics(predictions, cleaned_predictions, cleaned_ground_truths):
    metrics = {}
    partial_match_samples = []
    exact_match_samples = []

    valid_count = sum(1 for p in cleaned_predictions if p != "error")
    metrics['valid_predictions'] = valid_count / len(cleaned_predictions)

    exact_matches = 0
    for idx, (p, g) in enumerate(zip(cleaned_predictions, cleaned_ground_truths)):
        if p == g and p != "error":
            exact_matches += 1
            exact_match_samples.append({"index": idx, "prediction": p, "ground_truth": g})
    metrics['exact_match'] = exact_matches / len(cleaned_predictions)

    # BLEU Score (BLEU-1 with smoothing)
    bleu_scores = []
    smoother = SmoothingFunction().method1
    for pred, gt in zip(cleaned_predictions, cleaned_ground_truths):
        if pred != "error" and gt:
            try:
                pred_tokens = word_tokenize(pred)
                gt_tokens = word_tokenize(gt)
                if len(pred_tokens) > 0 and len(gt_tokens) > 0:
                    score = sentence_bleu([gt_tokens], pred_tokens, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoother)
                else:
                    score = 0
                bleu_scores.append(score)
            except:
                bleu_scores.append(0)
    metrics['bleu'] = sum(bleu_scores) / len(bleu_scores)

    # Partial Match (p in g or g in p)
    total_partial_matches = 0
    for idx, (p, g) in enumerate(zip(cleaned_predictions, cleaned_ground_truths)):
        if p != "error" and g and is_partial_match(p, g):
            total_partial_matches += 1
            partial_match_samples.append({"index": idx, "prediction": p, "ground_truth": g})
    metrics['partial_match'] = total_partial_matches / len(cleaned_predictions)

    # BERTScore
    filtered_preds = []
    filtered_refs = []
    for p, g in zip(predictions, cleaned_ground_truths):
        if p != "error" and g:
            filtered_preds.append(p)
            filtered_refs.append(g)

    P, R, F1 = bert_score(filtered_preds, filtered_refs, lang='en', verbose=True)
    metrics['bertscore_f1'] = F1.mean().item()

    pd.DataFrame(partial_match_samples).to_csv("partial_matches.csv", index=False)
    pd.DataFrame(exact_match_samples).to_csv("exact_matches.csv", index=False)

    print(f"\nExact Matches: {exact_matches}")
    print(f"Partial Matches (including exact): {total_partial_matches}")
    return metrics

# ─── Main Function ────────────────────────────────────────────────────
def main():
    csv_path = '/kaggle/input/totaldataset/simplified_vqa_dataset.csv'
    base_dir = '/kaggle/input/imagedataset'
    output_path = 'blip2_baseline_results.csv'

    processor, model = load_model()
    df = pd.read_csv(csv_path)

    predictions, raw_preds, cleaned_preds, cleaned_gts = process_dataset(df, base_dir, processor, model)

    df['raw_model_output'] = raw_preds
    df['model_prediction'] = predictions
    df['cleaned_prediction'] = cleaned_preds
    df['cleaned_ground_truth'] = cleaned_gts
    df.to_csv(output_path, index=False)

    metrics = evaluate_metrics(predictions, cleaned_preds, cleaned_gts)
    print("\nFinal Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    with open("metrics.txt", "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v:.4f}\n")

# ─── Run ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing in

2025-05-18 05:32:05.077795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747546325.510650      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747546325.636497      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/115 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/469 [00:00<?, ?it/s]

done in 21.16 seconds, 1418.16 sentences/sec

Exact Matches: 9437
Partial Matches (including exact): 13799

Final Metrics:
valid_predictions: 1.0000
exact_match: 0.3145
bleu: 0.4358
partial_match: 0.4598
bertscore_f1: 0.8862


In [1]:
!pip install bert_score

import os
import re
import gc
import string
import pandas as pd
import torch
from PIL import Image
from tqdm.notebook import tqdm
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from transformers import AutoProcessor, Blip2ForConditionalGeneration
from nltk.corpus import stopwords

# ─── NLTK Downloads ───────────────────────────────────────────────────
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

# ─── Global Stopword Set ──────────────────────────────────────────────
STOPWORDS = set(stopwords.words('english'))

# ─── Load BLIP-2 Model ────────────────────────────────────────────────
def load_model():
    processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16
    ).to("cuda")
    return processor, model

# ─── Text Cleaning ────────────────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().replace('_', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

# ─── Answer Extraction ────────────────────────────────────────────────
def extract_answer(text, question):
    if not text:
        return ""
    if isinstance(text, list):
        text = text[0]
    if "Answer:" in text:
        answer = text.split("Answer:")[1].strip()
    else:
        question_prefix = f"Question: {question}"
        answer = text.replace(question_prefix, "").strip() if question_prefix in text else text.strip()

    if question.lower().startswith(("is ", "are ", "does ", "can ", "was ", "will ", "should ")):
        if answer.lower().startswith("yes"):
            return "yes"
        elif answer.lower().startswith("no"):
            return "no"
    return answer

# ─── Inference Loop ───────────────────────────────────────────────────
def process_dataset(df, base_dir, processor, model):
    predictions = []
    raw_predictions = []
    cleaned_predictions = []
    cleaned_ground_truths = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            image = Image.open(os.path.join(base_dir, row['path'])).convert('RGB')
            question = row['generated_question']
            ground_truth = row['generated_answer']

            prompt = f"Question: {question} Answer:"
            inputs = processor(image, text=prompt, return_tensors="pt").to("cuda", torch.float16)

            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=10)

            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
            raw_answer = extract_answer(generated_text, question)
            cleaned_answer = clean_text(raw_answer)
            cleaned_gt = clean_text(ground_truth)

            predictions.append(raw_answer)
            raw_predictions.append(generated_text)
            cleaned_predictions.append(cleaned_answer)
            cleaned_ground_truths.append(cleaned_gt)

            if idx % 50 == 0:
                print(f"\n[{idx}] Q: {question}")
                print(f"Pred: {cleaned_answer} | GT: {cleaned_gt}")

        except Exception as e:
            print(f"Error @ {row['path']}: {e}")
            predictions.append("error")
            raw_predictions.append("error")
            cleaned_predictions.append("error")
            cleaned_ground_truths.append("error")

    return predictions, raw_predictions, cleaned_predictions, cleaned_ground_truths

# ─── Partial Match: pred in truth or truth in pred ────────────────────
def is_partial_match(pred, truth):
    return pred in truth or truth in pred

# ─── Metric Evaluation ────────────────────────────────────────────────
def evaluate_metrics(predictions, cleaned_predictions, cleaned_ground_truths):
    metrics = {}
    partial_match_samples = []
    exact_match_samples = []

    valid_count = sum(1 for p in cleaned_predictions if p != "error")
    metrics['valid_predictions'] = valid_count / len(cleaned_predictions)

    exact_matches = 0
    for idx, (p, g) in enumerate(zip(cleaned_predictions, cleaned_ground_truths)):
        if p == g and p != "error":
            exact_matches += 1
            exact_match_samples.append({"index": idx, "prediction": p, "ground_truth": g})
    metrics['exact_match'] = exact_matches / len(cleaned_predictions)

    # BLEU Score (BLEU-1 with smoothing)
    bleu_scores = []
    smoother = SmoothingFunction().method1
    for pred, gt in zip(cleaned_predictions, cleaned_ground_truths):
        if pred != "error" and gt:
            try:
                pred_tokens = word_tokenize(pred)
                gt_tokens = word_tokenize(gt)
                if len(pred_tokens) > 0 and len(gt_tokens) > 0:
                    score = sentence_bleu([gt_tokens], pred_tokens, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoother)
                else:
                    score = 0
                bleu_scores.append(score)
            except:
                bleu_scores.append(0)
    metrics['bleu'] = sum(bleu_scores) / len(bleu_scores)

    # Partial Match (p in g or g in p)
    total_partial_matches = 0
    for idx, (p, g) in enumerate(zip(cleaned_predictions, cleaned_ground_truths)):
        if p != "error" and g and is_partial_match(p, g):
            total_partial_matches += 1
            partial_match_samples.append({"index": idx, "prediction": p, "ground_truth": g})
    metrics['partial_match'] = total_partial_matches / len(cleaned_predictions)

    # BERTScore
    filtered_preds = []
    filtered_refs = []
    for p, g in zip(predictions, cleaned_ground_truths):
        if p != "error" and g:
            filtered_preds.append(p)
            filtered_refs.append(g)

    P, R, F1 = bert_score(filtered_preds, filtered_refs, lang='en', verbose=True)
    metrics['bertscore_f1'] = F1.mean().item()

    pd.DataFrame(partial_match_samples).to_csv("partial_matches.csv", index=False)
    pd.DataFrame(exact_match_samples).to_csv("exact_matches.csv", index=False)

    print(f"\nExact Matches: {exact_matches}")
    print(f"Partial Matches (including exact): {total_partial_matches}")
    return metrics

# ─── Main Function ────────────────────────────────────────────────────
def main():
    csv_path = '/kaggle/input/trydataset/sampled_vqa_dataset.csv'
    base_dir = '/kaggle/input/imagedataset'
    output_path = 'blip2_baseline_results.csv'

    processor, model = load_model()
    df = pd.read_csv(csv_path)

    predictions, raw_preds, cleaned_preds, cleaned_gts = process_dataset(df, base_dir, processor, model)

    df['raw_model_output'] = raw_preds
    df['model_prediction'] = predictions
    df['cleaned_prediction'] = cleaned_preds
    df['cleaned_ground_truth'] = cleaned_gts
    df.to_csv(output_path, index=False)

    metrics = evaluate_metrics(predictions, cleaned_preds, cleaned_gts)
    print("\nFinal Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    with open("metrics.txt", "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v:.4f}\n")

# ─── Run ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing in

2025-05-18 11:06:06.440546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747566366.670438      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747566366.738847      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 0.67 seconds, 743.37 sentences/sec

Exact Matches: 173
Partial Matches (including exact): 244

Final Metrics:
valid_predictions: 1.0000
exact_match: 0.3460
bleu: 0.4611
partial_match: 0.4880
bertscore_f1: 0.8889
